# CPDat v4.0 cleanup

Cleans the [CPDat](https://doi.org/10.23645/epacomptox.5352997.v5) (Chemical and Products Database) v4.0 product composition data so that it contains individual chemicals with standardized identifiers and harmonized product use categories (PUCs). CPDat is a collection of commercial use information for thousands of chemicals.

**Workflow**

| Step | What happens |
| --- | --- |
| 1 | Load the CPDat product composition CSV |
| 2 | Drop rows with no values in both `cas` and `name` |
| 3 | Check `product_title` values and replace if missing |
| 4 | Retrieve InChIKeys and SMILES from PubChem (built in; resumable, cached) |
| 5 | Move columns and create the `weight_fraction` column; save the intermediate table |
| 6 | Remove compounds (no InChIKey, specific InChIKeys, unwanted keywords) |
| 7 | Shorten & rename `puc_general_category` values |
| 8 | Populate `puc_product_type` missing values |
| 9 | Additional PUC columns cleanup |
| 10 | Fill missing PUC values (`Unknown` / `NoID`) and save |
| 11 | Create the `puc_harmonized` column, sort and save |
| 12 | Show descriptive statistics and save summary tables |

**Settings**

All user-editable settings are in the configuration cell:

- `CSV_PATH`: the raw CPDat product composition CSV (`cpdat_v4.0_product_composition_data.csv`; see https://doi.org/10.23645/epacomptox.5352997.v5).
- `OUTPUT_DIR` / `CACHE_DIR`: output (`cpdat_cleaned`) and cache (`cache`) folders, created next to the notebook.
- `LOOKUP_LIMIT`: set to e.g. 50 for a quick test run of the PubChem lookups; `None` runs all compounds.
- `MAX_REQ_PER_SEC` and `N_WORKERS`: PubChem request rate (PubChem requests <= 5 requests/second) and number of parallel workers.

**Requirements:** Python 3 with `pandas`, `numpy` and `requests`, the shared `inquire_common` folder (utilities, PubChem client and filter lists) in the parent folder of this notebook, and internet access for the PubChem step.

**Output files** (written to `cpdat_cleaned/`)

| File | Description |
| --- | --- |
| `cpdat_v4.0_products_PubChem_Fltrd.csv` | Cleaned table with PubChem data, before InChIKey/keyword filtering |
| `cpdat_v4.0_products_PubChem_KeyFltrd.csv` | After InChIKey/keyword filtering |
| `cpdat_v4.0_products_PubChem_KeyFltrd_ProdType.csv` | After PUC cleanup and `puc_product_type` population |
| `cpdat_v4.0_products_PubChem_KeyFltrd_PUC_harm.csv` | Final table with the `puc_harmonized` column |
| `cpdat_puc_summary.csv` | Unique values of the PUC columns |
| `cpdat_cas_product_composition_summary.csv` | Per-compound statistics (one row per `cas`) |

In [1]:
# Reset env to erase previous vars, dicts, dfs, etc.
%reset -f

from datetime import datetime

print(f"Environment reset | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

Environment reset | 2026-11-18 12:33:12



In [ ]:
# Configuration & imports
import os
import re
import time
from collections import defaultdict, OrderedDict
from datetime import datetime

import numpy as np
import pandas as pd

# Shared helpers/filters (common lives in the parent folder of this notebook)
import sys
sys.path.insert(0, os.path.abspath(".."))
from common.utils import print_overwrite, df_stats
from common.compound_filters import apply_standard_filters
from common.pubchem import PubChemClient, lookup_attempts, per_row, run_cached_lookup
from common.cleaning import add_uid, read_cache

# CPDat cleanup rule tables (rules/*.csv) and the shared loader (inquire_common/rules.py)
from common.rules import RuleSet
rules = RuleSet("rules")

# ---------------------------------------------------------------------------
# CONFIGURATION (all user-editable settings live here)
# ---------------------------------------------------------------------------
# Input: CPDat v4.0 product composition CSV (downloaded from https://doi.org/10.23645/epacomptox.5352997.v5)
CSV_PATH = "../original_databases/cpdat_v4.0/cpdat_v4.0_product_composition_data.csv"

# Output & cache folders (subfolders of the notebook folder)
BASE_DIR = "."
OUTPUT_DIR = os.path.join(BASE_DIR, "cpdat_cleaned")
CACHE_DIR = os.path.join(BASE_DIR, "cache")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# Cache file (resumable PubChem lookups, one row per unique identifier set)
PUBCHEM_CACHE_CSV = os.path.join(CACHE_DIR, "PubChem_lookup_cache.csv")

# Output files
INTERMEDIATE_CSV = os.path.join(OUTPUT_DIR, "cpdat_v4.0_products_PubChem_Fltrd.csv")                 # cleaned table before InChIKey/keyword filtering (step 5)
KEYFLTRD_CSV = os.path.join(OUTPUT_DIR, "cpdat_v4.0_products_PubChem_KeyFltrd.csv")                  # after InChIKey/keyword filtering (step 6)
PRODTYPE_CSV = os.path.join(OUTPUT_DIR, "cpdat_v4.0_products_PubChem_KeyFltrd_ProdType.csv")         # after PUC cleanup (step 10)
HARMONIZED_CSV = os.path.join(OUTPUT_DIR, "cpdat_v4.0_products_PubChem_KeyFltrd_PUC_harm.csv")       # final table with puc_harmonized (step 11)
PUC_SUMMARY_CSV = os.path.join(OUTPUT_DIR, "cpdat_puc_summary.csv")                                  # unique values of the PUC columns (step 12)
STATS_CSV = os.path.join(OUTPUT_DIR, "cpdat_cas_product_composition_summary.csv")                    # per-compound statistics (step 12)

# PubChem enrichment settings
LOOKUP_LIMIT = None      # set to e.g. 50 for a quick test run; None = all compounds
MAX_REQ_PER_SEC = 4      # PubChem asks for <= 5 requests/second. You may be blocked when increasing the rate.
N_WORKERS = 4
CHUNK = 200              # compounds per cache write; smaller chunks give a more responsive ETA

print(f"Configuration & imports done | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Configuration & imports done | 2026-11-18 12:34:00



In [ ]:
# Helper to remove previous progress message
# print_overwrite is imported from inquire_common.utils in the configuration cell

print(f"Helper to remove previous progress message created | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

## Step 1: Load the CPDat CSV

Paths and settings are defined in the configuration cell above. Confirm that the CSV exists before reading it. The raw product composition file has no InChIKey or SMILES columns; these are added from PubChem in step 4.

In [ ]:
# Check that the CPDat CSV exists (CSV_PATH is set in the configuration cell)
assert os.path.exists(CSV_PATH), f"CSV file not found: {CSV_PATH}"
print("CSV path is valid.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

Read the CSV into `df`. The file is large (ca. 3.4 million rows), so this takes a while.

In [ ]:
# Read CSV file
print("Reading CSV with pandas.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print("It will take a while...\n")

# Read entire CSV
df = pd.read_csv(CSV_PATH, low_memory=False)

# Standard schema: name = curated chemical name, orig_name = raw chemical name, cas = curated CASRN
df = df.rename(columns={"curated_chemical_name": "name", "raw_chemical_name": "orig_name", "curated_casrn": "cas"})

print("CSV loaded.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print(f"\nCSV shape: \n{df.shape[0]:,} rows × {df.shape[1]} columns\n")

## Step 2: Drop rows without chemical identifiers

Drops rows with no values in both `cas` and `name`. This is done before the PubChem step, so that only compounds that can be looked up are queried.

In [ ]:
# Drop rows with no values in both cas and name
print("Dropping rows with no values in 'cas' & 'name' columns.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")
print(f"DF shape before action: \n{df.shape[0]:,} rows × {df.shape[1]} columns\n")

df_drop = df.dropna(
    subset=["cas", "name"],
    how="all"
).copy()
row_diff = (len(df)-len(df_drop))
print(f"{row_diff:,} rows with no values in 'cas' & 'name' columns were dropped.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")
print(f"Rows after filtering: \n{len(df_drop):,}\n")


## Step 3: Check and fill `product_title`

Checks the `product_title` column for missing or odd values (`MISSING_VALUES = ["", " ", "-", "—", "–", "NA", "NaN", "nan", "N/A", "n/a"]`), after stripping and collapsing whitespace. Missing titles are filled first from `data_document_title`, then from `product_upc`. Each replacement is printed.

In [ ]:
# Check product_title and replace if missing/NaN/NA/etc.
print("Checking 'product_title' column for missing/odd values.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Keep original product titles for diagnostics
df_drop["product_title_original"] = df_drop["product_title"]

# Define what counts as "missing"
MISSING_VALUES = ["", " ", "-", "-", "—", "–", "NA", "NaN", "nan", "N/A", "n/a"]

# Normalize product_title: 
# Convert placeholders to NaN, strip leading/trailing spaces, collapse multiple spaces
df_drop["product_title"] = (
    df_drop["product_title"]
    .astype("string")
    .str.strip()                               # remove leading/trailing spaces
    .str.replace(r"\s+", " ", regex=True)      # collapse multiple spaces
    .replace(MISSING_VALUES, np.nan)
)

# Fill missing product_title
# First from data_document_title, then from product_upc columns
fill_from_doc = (
    df_drop["product_title"].isna() &
    df_drop["data_document_title"].notna()
)

# Second fallback: product_upc
fill_from_upc = (
    df_drop["product_title"].isna() &
    df_drop["product_upc"].notna()
)

df_drop.loc[fill_from_doc, "product_title"] = (
    df_drop.loc[fill_from_doc, "data_document_title"]
    .astype("string")
    .str.strip()
)
df_drop.loc[fill_from_upc, "product_title"] = (
    df_drop.loc[fill_from_upc, "product_upc"]
    .astype("string")
    .str.strip()
)

# Print product titles before and after replacement
print("Product title replacements:")

replaced_rows = df_drop[
    df_drop["product_title_original"].replace(MISSING_VALUES, np.nan).isna() &
    df_drop["product_title"].notna()
]

for _, r in replaced_rows.iterrows():
    old_val = r["product_title_original"]
    new_val = r["product_title"]

    old_val_str = repr(old_val) if pd.notna(old_val) else "NaN"
    new_val_str = repr(new_val)

    print(f"{old_val_str} >>> {new_val_str}")

# Drop product_title_original column
df_drop = df_drop.drop(columns=['product_title_original'])

# Diagnostics
print("\nProduct title check & replacement completed.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")
print("Summary:")
print(f"Total rows: {len(df_drop):,}")
print(f"Replaced product_title values: {len(replaced_rows):,}")
print(f"Remaining missing 'product_title' values: {df_drop['product_title'].isna().sum():,}\n")

## Step 4: Retrieve InChIKeys and SMILES from PubChem

The original workflow ran the (cleaned) CPDat table through the external `PubChem_retriever/PubChem_Retriever.py` script before this notebook, to add `InChIKey` and `SMILES` columns (saved as `cpdat_v4.0_product_composition_data_PubChem.csv`). This step replaces that manual step with the shared PubChem client from `inquire_common/pubchem.py`, as in the FooDB, SPECIATE and SPECIEUROPE notebooks: requests go to the PubChem PUG REST API, are rate-limited (`MAX_REQ_PER_SEC`, with extra delay if PubChem's `X-Throttling-Control` header reports load), run in parallel (`N_WORKERS`) and are written to a cache file as they come in, so the step can be interrupted and re-run.

Each distinct combination of `cas`, `name` and `dtxsid` gets a hash-based `uid`. For each one the lookup tries, in order, until PubChem returns a hit:

1. `cas` (PubChem resolves CAS numbers as synonyms)
2. `dtxsid` (also resolved as a synonym)
3. `name`

The method that succeeded is stored in `Lookup_method`. Compounds PubChem cannot resolve are cached as `not_found`, so they are not retried on re-runs (delete their rows from the cache to retry). Compounds that fail because of a transient error are not cached and are retried on the next run.

**Total time to arrival** for approximately 12800 retrievals: ~ 1.5--2 hr.

**Caveats**: *the original script looked up the name first and then the InChIKey; the order here (CAS, DTXSID, name) is a choice, not a reproduction of the original. The first hit is taken when a query matches several PubChem compounds. Set `LOOKUP_LIMIT` in the configuration cell for a quick test run.*

The first cell defines the helper functions; the second runs the lookup.

In [ ]:
# PubChem client (rate limiter and request helpers live in inquire_common/pubchem.py)
client = PubChemClient(max_req_per_sec=MAX_REQ_PER_SEC, props="Title,IUPACName,InChIKey,SMILES")

def lookup_compound(row):
    """row: (uid, casrn, dtxsid, name). Tries CAS#, DTXSID and name in order."""
    uid, casrn, dtxsid, name = row
    return lookup_attempts(uid, [
        ("CAS", casrn, client.name_lookup_row),
        ("DTXSID", dtxsid, client.name_lookup_row),
        ("Name", name, client.name_lookup_row),
    ])

print(f"PubChem helpers defined | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# Retrieve PubChem data for the unique identifier sets (resumable; results are appended to the cache)
print("Creating unique 'cas' & 'dtxsid' & 'name' set for PubChem data retrieval.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Distinct identifier sets; the uid is a hash of the values, so the cache stays valid if row order changes
UNIQUE_KEY_COLS = ["cas", "dtxsid", "name"]

df_unique = add_uid(df_drop, UNIQUE_KEY_COLS)
print(f"Unique identifier sets: {len(df_unique):,}\n")

cache_cols = ["uid", "CID", "InChIKey_PubChem", "SMILES_PubChem", "Lookup_method"]
cache_current = run_cached_lookup(
    df_unique, UNIQUE_KEY_COLS, per_row(lookup_compound, N_WORKERS),
    PUBCHEM_CACHE_CSV, cache_cols, client, chunk=CHUNK, limit=LOOKUP_LIMIT,
)


Merge the PubChem data into the CPDat table (`df_drop`) using the three identifier columns. Columns added (named as in the original `_PubChem.csv` file where applicable):

| Column | Content |
| --- | --- |
| `inchikey` | PubChem InChIKey |
| `smiles` | PubChem SMILES |
| `pubchem_cid` | PubChem compound ID |
| `pubchem_lookup_method` | `CAS`, `DTXSID`, `Name` or `not_found` |

`validate="m:1"` makes the merge fail if the lookup table were not unique per identifier set, and the row count is asserted to be unchanged. Compounds that PubChem could not resolve keep missing values and are removed in step 6.

In [ ]:
# Merge the PubChem results (cache from the lookup above) back onto the full table
print("Merging PubChem data into CPDat.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

pubchem = (
    read_cache(PUBCHEM_CACHE_CSV)
    .rename(columns={
        "InChIKey_PubChem": "inchikey",
        "SMILES_PubChem": "smiles",
        "CID": "pubchem_cid",
        "Lookup_method": "pubchem_lookup_method",
    })
)

# One row per unique identifier set, now with the PubChem columns
df_unique_pc = df_unique.merge(pubchem, on="uid", how="left", validate="1:1")
n_missing_lookup = df_unique_pc["pubchem_lookup_method"].isna().sum()
if n_missing_lookup:
    print(f"WARNING: {n_missing_lookup:,} unique compounds have no cached result yet. Re-run the lookup cell to retrieve them.\n")

# Many-to-one join back onto the full table
n_before = len(df_drop)
df_drop = df_drop.merge(
    df_unique_pc.drop(columns="uid"),
    on=UNIQUE_KEY_COLS,
    how="left",
    validate="m:1",
)
assert len(df_drop) == n_before, "Row count changed during the PubChem merge"

print(f"Rows with InChIKey: {df_drop['inchikey'].notna().sum():,} of {len(df_drop):,}")
print(f"\nCPDat + PubChem shape: \n{df_drop.shape[0]:,} rows × {df_drop.shape[1]} columns\n")
print("Done merging PubChem data.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

## Step 5: Move columns, create `weight_fraction` and save the intermediate table

1. Moves the PubChem columns (`inchikey`, `smiles`, `pubchem_cid`, `pubchem_lookup_method`) to right before `dtxsid`, and `component_name` to right after `puc_classification_method`.
2. Creates the `weight_fraction` column (placed right after `central_weight_fraction`): the midpoint of `lower_weight_fraction` and `upper_weight_fraction` when both are present; otherwise `central_weight_fraction`; otherwise whichever of lower/upper is present.
3. Writes the table to CSV (`INTERMEDIATE_CSV`), before InChIKey/keyword filtering.

In [ ]:
# Move columns to specific positions
print("Moving columns.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

cols = list(df_drop.columns)

pubchem_cols = ["inchikey", "smiles", "pubchem_cid", "pubchem_lookup_method"]
for c in pubchem_cols:
    cols.remove(c)

dtxsid_idx = cols.index("dtxsid")
cols[dtxsid_idx:dtxsid_idx] = pubchem_cols

df_drop = df_drop[cols].copy()

cols = list(df_drop.columns)

cols.remove("component_name")
pcm_idx = cols.index("puc_classification_method")

cols.insert(pcm_idx + 1, "component_name")

df_move = df_drop[cols].copy()

print("Done moving columns.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


In [ ]:
# Create weight_fraction column
print("Creating 'weight_fraction' column with midpoint values.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

lwf = df_move["lower_weight_fraction"]
uwf = df_move["upper_weight_fraction"]
cwf = df_move["central_weight_fraction"]

df_move["weight_fraction"] = np.where(
    lwf.notna() & uwf.notna(),
    (lwf + uwf) / 2,
    np.where(
        cwf.notna(),
        cwf,
        np.where(
            lwf.notna(),
            lwf,
            uwf
        )
    )
)

cols = list(df_move.columns)

cols.remove("weight_fraction")
cwf_idx = cols.index("central_weight_fraction")

cols.insert(cwf_idx + 1, "weight_fraction")

df_wf = df_move[cols].copy()

print("Created 'weight_fraction' column with midpoint values.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


In [ ]:
# Write filtered CPDat to CSV file
print("Writing filtered CPDat to CSV.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Save CPDat BEFORE the InChIKey, weight_fraction & name/keyword filtration
df_wf.to_csv(INTERMEDIATE_CSV, index=False)

print(f"Filtered DF written to {INTERMEDIATE_CSV}.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

## Step 6: Remove non-individual-chemical entries

Filters the table in three stages, each printing what was dropped:

1. **No InChIKey:** rows without an InChIKey are removed (mixtures or compounds PubChem could not resolve to a single structure).
2. **Specific InChIKeys:** rows whose InChIKey is in the shared list `inquire_common/filters/drop_inchikeys.txt` are removed (elements, inorganic ions and salts, acids, and similar). The trailing comments in the list give the compound each key stands for.
3. **Keywords in `name`:** rows whose name matches a keyword (metals, minerals, inorganic compounds, mixtures and biological materials) are removed. The keywords come from `keywords_common.txt` (shared by all sources) plus `keywords_cpdat.txt`, minus the protected names in `keyword_exceptions.txt`. Matching is case-insensitive and on whole words (`anchor_end=True`), as in the original CPDat notebook.

These lists replace the lists previously hard-coded in this notebook; the shared lists contain all of the former entries. The keyword and InChIKey lists are curated by hand, so review them against your own scope before relying on the output.

The filtered table is then sorted by `product_title` and `weight_fraction` and written to CSV (`KEYFLTRD_CSV`). The previous version sorted by `weight_fraction` and then by `product_title` in two separate calls, so only the last call determined the order; a single multi-key stable sort is used now.

In [ ]:
# Filter database by InChIKey & keywords
print("Filtering CPDat by InChIKey & other keywords.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Lists live in inquire_common/filters: drop_inchikeys.txt, keywords_common.txt + keywords_cpdat.txt,
# keyword_exceptions.txt. anchor_end=True keeps CPDat's original whole-word matching.
initial_n = len(df_wf)
df_inchikey, dropped_df, dropped_chems = apply_standard_filters(
    df_wf, source="cpdat", anchor_end=True, label="CPDat",
)

print("Done keyword-filtering CPDat.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


In [ ]:
# Write keyword-filtered CPDat to CSV file (path set in the configuration cell)
print("Sorting keyword-filtered CPDat.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Sort by both keys at once (first key is the primary key, later keys break ties)
df_inchikey = df_inchikey.sort_values(["product_title", "weight_fraction"], kind="stable")

print("Writing keyword-filtered CPDat to CSV.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Save CPDat AFTER the InChIKey, weight_fraction & name/keyword filtration
df_inchikey.to_csv(KEYFLTRD_CSV, index=False)

print(f"Keyword-filtered CPDat written to {KEYFLTRD_CSV}.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")
print(f"Keyword-filtered CPDat CSV shape: \n{df_inchikey.shape[0]:,} rows × {df_inchikey.shape[1]} columns\n")

## Step 7: Shorten and rename `puc_general_category` values

Maps the long general product use category names to short names (e.g., `Pet and animal care` to `Animal Care`, `Cleaning products and household care` to `Cleaning - Household`). The resulting unique values are printed.

In [ ]:
# Shorten & rename puc_general_category values
print("Renaming 'puc_general_category'.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

df_puc_gen = df_inchikey

# old -> new category names, loaded from rules/puc_general_map.csv
PUC_GENERAL_MAP = rules.load_map("puc_general_map")

df_puc_gen["puc_general_category"] = (
    df_puc_gen["puc_general_category"]
    .replace(PUC_GENERAL_MAP)
)

print("New 'puc_general_category' column values:")
for val in sorted(df_puc_gen['puc_general_category'].dropna().unique()):
    print(val)
print()

print("Done renaming 'puc_general_category'.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


## Step 8: Populate `puc_product_type` missing values

Missing `puc_product_type` values are populated in this order:

1. Extract a use-related keyword from `puc_product_family`, based on `puc_product_family` to `product_title` keyword matching.
2. Extract a use-related keyword from `product_title`.
3. Fall back to the `puc_product_family` value.
4. Final fallback to the `puc_general_category` value.
5. Infer an unknown keyword when all four PUC columns are missing.

The first two cells define the helper functions and the CPDat-informed keyword map (`USE_KEYWORDS`); the third applies them.

In [ ]:
# Helpers to populate puc_product_type missing values

# Helper functions
STOPWORDS = {"and", "or", "for", "at", "in", "on", "with", "of", "to", "from"}

ABBREVIATIONS = {
    "adh": "adhesive",
    "adhes.": "adhesive",
    "adhsv": "adhesive",
    "activ": "activatior",
    "solv": "solvent",
    "chlor stabiliz": "chlorine stabilizer"
}

def normalize_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"[,_\-()/™®]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def singularize(word):
    # very conservative plural → singular conversion
    if word.endswith("ies"):
        return word[:-3] + "y"
    if word.endswith("ses"):
        return word[:-2]
    if word.endswith("s") and not word.endswith("ss"):
        return word[:-1]
    return word

def normalize_multi_terms(x: str) -> str:
    if not isinstance(x, str):
        return x

    s = x.lower().strip()

    # normalize separators to " or "
    s = re.sub(r"\s*/\s*", " or ", s)
    s = re.sub(r"\s+and\s+", " or ", s)

    # split, singularize, deduplicate
    parts = [p.strip() for p in s.split(" or ") if p.strip()]
    parts = [canonicalize_type(p) for p in parts]

    return " or ".join(sorted(set(parts)))

def replace_and_with_ampersand(x):
    if not isinstance(x, str):
        return x
    return re.sub(r"\band\b", "&", x, flags=re.IGNORECASE)

def smart_title(x: str) -> str:
    if not isinstance(x, str):
        return x

    words = re.split(r"(\s+)", x.lower())
    out = []

    for w in words:
        if w.strip() in STOPWORDS or not w.strip().isalpha():
            out.append(w)
        else:
            out.append(w.capitalize())

    return "".join(out)

def normalize_apostrophes(x):
    if not isinstance(x, str):
        return x
    return x.replace("'", "’")

def extract_keywords_from_family(family):
    family = normalize_text(family)
    words = []
    for w in family.split():
        if w in STOPWORDS:
            continue
        w = ABBREVIATIONS.get(w, w)
        words.append(singularize(w))
    return list(OrderedDict.fromkeys(words))  # preserve order, unique

def keyword_found(keyword, title):
    title = normalize_text(title)
    return re.search(rf"\b{re.escape(keyword)}\b", title) is not None

CANONICAL_MAP = {
    frozenset(["adhesive", "remover"]): "adhesive remover",
    frozenset(["oil", "paint"]): "oil-based paint",
    frozenset(["wall", "covering", "paste"]): "wall covering paste",
}

def canonicalize_type(value):
    if pd.isna(value):
        return value

    parts = [singularize(p.strip()) for p in value.split(";")]
    key = frozenset(parts)

    if key in CANONICAL_MAP:
        return CANONICAL_MAP[key]

    # fallback: single clean label
    return singularize(parts[0])

def match_use_keywords(title):
    title_norm = normalize_text(title)
    matches = []

    for label, phrases in USE_KEYWORDS.items():
        for p in phrases:
            p_norm = normalize_text(p)
            if re.search(rf"\b{re.escape(p_norm)}\b", title_norm):
                matches.append(label)
                break
    return matches

def all_puc_missing(row):
    return all(
        is_missing(row[col])
        for col in [
            "puc_kind",
            "puc_general_category",
            "puc_product_family",
            "puc_product_type",
        ]
    )

def fix_mojibake(text):
    if not isinstance(text, str):
        return text
    return (
        text
        .replace("���", "’")
        .replace("â€™", "’")
        .replace("â€˜", "‘")
    )

def replace_full_cell(value, mapping):
    if not isinstance(value, str):
        return value
    key = value.strip().lower()
    return mapping.get(key, value)

# Context-dependent replacements of puc_product_type, loaded from rules/conditional_puc_type_rules.csv.
# A rule applies when the (lower-cased) type equals `puc_product_type` and `contains` occurs in the
# lower-cased `match_column`; within one type the first matching rule (file order) wins.
CONDITIONAL_PUC_RULES = rules.load_conditional_rules()

def conditional_puc_type(row):
    val = row["puc_product_type"]
    if not isinstance(val, str):
        return val

    rules = CONDITIONAL_PUC_RULES.get(val.lower())
    if rules:
        context = {
            col: str(row.get(col, "")).lower()
            for col in ("puc_general_category", "puc_product_family", "product_title")
        }
        for column, contains, new_value in rules:
            if contains in context[column]:
                return new_value

    return val

print("Helper functions loaded.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

In [ ]:
# Rule tables (CPDat-informed keyword maps & value maps)
# Rules live in rules/*.csv next to this notebook; edit those files to change a mapping (see rules/README.md).
# Row order is preserved; it matters for UNKNOWN_KEYWORDS (first match wins), not for the other maps here.
USE_KEYWORDS = rules.load_use_keywords("use_keywords")  # label -> [phrases]
PUC_TYPE_EXACT_MAP = rules.load_map("puc_type_exact_map")
PUC_TYPE_FINAL_REMAP = rules.load_map("puc_type_final_remap")
UNKNOWN_KEYWORDS = rules.load_map("unknown_keywords")
PUC_TYPE_FINAL_BRUSH_MAP = rules.load_map("puc_type_final_brush_map")
PUC_TYPE_FULL_CELL_MAP = rules.load_map("puc_type_full_cell_map")
PUC_PRODUCT_TYPE_FINAL_MANUAL = rules.load_map("puc_product_type_final_manual")
PUC_PRODUCT_FAMILY_FINAL_MANUAL = rules.load_map("puc_product_family_final_manual")
PUC_GENERAL_CATEGORY_FINAL_MANUAL = rules.load_map("puc_general_category_final_manual")

print("Dictionaries loaded.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


In [ ]:
# Populate puc_product_type missing values
print("Checking 'puc_product_type' for missing values & replacing those.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

df_prod_type = df_puc_gen

# Track edits to puc_product_type
if "edited" not in df_prod_type.columns:
    df_prod_type["edited"] = 0

# Move 'edited' column after puc_product_type
cols = list(df_prod_type.columns)
if "edited" in cols and "puc_product_type" in cols:
    cols.insert(cols.index("puc_product_type") + 1, cols.pop(cols.index("edited")))
    df_prod_type = df_prod_type[cols].copy()

print("Created 'edited' column.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

print("Starting main 'puc_product_type' checkup.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Main logic: 1a → 1b → 1c
def is_missing(x):
    return pd.isna(x) or str(x).strip() == ""

def missing_mask(s):
    # Vectorized equivalent of is_missing()
    return s.isna() | (s.astype(str).str.strip() == "")

# Precompiled use-keyword patterns (built once instead of once per row)
USE_PATTERNS = {
    label: [re.compile(rf"\b{re.escape(normalize_text(p))}\b") for p in phrases]
    for label, phrases in USE_KEYWORDS.items()
}

def match_use_keywords_fast(title_norm):
    return [
        lab for lab, pats in USE_PATTERNS.items()
        if any(p.search(title_norm) for p in pats)
    ]

def infer_type(title, family):
    title_norm = normalize_text(title)
    new_value = None

    # 1a: family → title keyword matching
    if not is_missing(family):
        found = [
            kw for kw in extract_keywords_from_family(family)
            if re.search(rf"\b{re.escape(kw)}\b", title_norm)
        ]
        if found:
            new_value = "; ".join(sorted(set(found)))

    # 1b: use-related keyword matching
    if new_value is None:
        found = match_use_keywords_fast(title_norm)
        if found:
            new_value = "; ".join(sorted(set(found)))

    # 1c: fallback to family
    if new_value is None and not is_missing(family):
        new_value = family

    return canonicalize_type(new_value) if new_value is not None else None

# The logic depends only on (product_title, puc_product_family), and each product
# repeats once per chemical -> evaluate once per unique pair, then merge back.
key_cols = ["product_title", "puc_product_family"]
miss = missing_mask(df_prod_type["puc_product_type"])
pairs = df_prod_type.loc[miss, key_cols].drop_duplicates()
print(f"{miss.sum():,} rows with missing 'puc_product_type' -> {len(pairs):,} unique (title, family) pairs")

pairs["new_type"] = [
    infer_type(t, f) for t, f in zip(pairs["product_title"], pairs["puc_product_family"])
]

filled = df_prod_type.loc[miss, key_cols].merge(pairs, on=key_cols, how="left")
filled.index = df_prod_type.index[miss]
ok = filled["new_type"].notna()
df_prod_type.loc[filled.index[ok], "puc_product_type"] = filled.loc[ok, "new_type"]
df_prod_type.loc[filled.index[ok], "edited"] = 1

print("Done with main 'puc_product_type' checkup.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Final fallback: puc_general_category → puc_product_type
mask = (
    df_prod_type["puc_product_type"].isna()
    & df_prod_type["puc_general_category"].notna()
)
    
df_prod_type.loc[mask, "puc_product_type"] = df_prod_type.loc[mask, "puc_general_category"]
df_prod_type.loc[mask, "edited"] = 1

print("Populated 'puc_product_type' with 'puc_general_category' values.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# UNKNOWN keyword inference when all four PUC cols are missing
puc4 = ["puc_kind", "puc_general_category", "puc_product_family", "puc_product_type"]
all_missing = pd.concat([missing_mask(df_prod_type[c]) for c in puc4], axis=1).all(axis=1)

def infer_unknown(title):
    title = str(title).lower()
    for kw, val in UNKNOWN_KEYWORDS.items():
        if kw in title:
            return val
    return None

unk_titles = df_prod_type.loc[all_missing, "product_title"]
unk_vals = unk_titles.map({t: infer_unknown(t) for t in unk_titles.unique()})
hit = unk_vals.notna()
df_prod_type.loc[unk_vals.index[hit], "puc_product_type"] = unk_vals[hit]
df_prod_type.loc[unk_vals.index[hit], "edited"] = 1

print("Inferred UNKNOWN keyword when all four PUC cols are missing.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Exact full-cell cleanup for puc_product_type
df_prod_type["puc_product_type"] = (
    df_prod_type["puc_product_type"]
    .apply(lambda x: PUC_TYPE_EXACT_MAP.get(x.lower(), x) if isinstance(x, str) else x)
)

print("Done with full-cell cleanup for 'puc_product_type'.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Singularization / canonical cleanup
df_prod_type["puc_product_type"] = df_prod_type["puc_product_type"].apply(
    lambda x: canonicalize_type(x) if isinstance(x, str) else x
)

print("Done with singularization / canonical cleanup.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Singularize + normalize separators
df_prod_type["puc_product_type"] = df_prod_type["puc_product_type"].apply(
    lambda x: normalize_multi_terms(x) if isinstance(x, str) else x
)

print("Done with singularization & separators normalization.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Replace “and” with “&” (family + type)
for col in ["puc_product_family", "puc_product_type"]:
    if col in df_prod_type.columns:
        df_prod_type[col] = df_prod_type[col].apply(replace_and_with_ampersand)

print("Done with replacing “and” with “&”.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Apostrophe normalization
for col in ["puc_general_category", "puc_product_family", "puc_product_type"]:
    if col in df_prod_type.columns:
        df_prod_type[col] = df_prod_type[col].apply(normalize_apostrophes)

print("Done with apostrophe normalization.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Fix children���s (mojibake) to proper Unicode apostrophe
for col in ["puc_general_category", "puc_product_family", "puc_product_type"]:
    if col in df_prod_type.columns:
        df_prod_type[col] = df_prod_type[col].apply(fix_mojibake)

print("Done with mojibake normalization.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")
    
# Final category cleanup
df_prod_type["puc_product_type"] = df_prod_type["puc_product_type"].apply(
    lambda x: PUC_TYPE_FINAL_REMAP.get(x.lower(), x) if isinstance(x, str) else x
)

print("Done with final category cleanup.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Final unconditional replacements (phrase-level)
df_prod_type["puc_product_type"] = df_prod_type["puc_product_type"].apply(
    lambda x: PUC_TYPE_FINAL_BRUSH_MAP.get(x.lower(), x)
    if isinstance(x, str) else x
)

print("Done with unconditional replacements (phrase-level).", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Full-cell-only replacements (exact match)
df_prod_type["puc_product_type"] = df_prod_type["puc_product_type"].apply(
    lambda x: replace_full_cell(x, PUC_TYPE_FULL_CELL_MAP)
)

print("Done with full-cell-only replacements (exact match).", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Conditional replacements (context-dependent)
# Result depends only on these four columns -> evaluate once per unique combination
cond_cols = ["puc_product_type", "puc_general_category", "puc_product_family", "product_title"]
cond_uniq = df_prod_type[cond_cols].drop_duplicates().reset_index(drop=True)
cond_uniq["_new"] = cond_uniq.apply(conditional_puc_type, axis=1)
cond_merged = df_prod_type[cond_cols].merge(cond_uniq, on=cond_cols, how="left")
df_prod_type["puc_product_type"] = cond_merged["_new"].to_numpy()

print("Done with conditional replacements (context-dependent).", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Convert selected PUC columns to Title Case
for col in ["puc_kind", "puc_product_family", "puc_product_type"]:
    if col in df_prod_type.columns:
        df_prod_type[col] = df_prod_type[col].apply(smart_title)

print("Done converting selected PUC columns to Title Case.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Final cleanup
df_prod_type["puc_product_type"] = df_prod_type["puc_product_type"].apply(
    lambda x: PUC_PRODUCT_TYPE_FINAL_MANUAL.get(x, x)
    if isinstance(x, str) else x
)

df_prod_type["puc_product_family"] = df_prod_type["puc_product_family"].apply(
    lambda x: PUC_PRODUCT_FAMILY_FINAL_MANUAL.get(x, x)
    if isinstance(x, str) else x
)

df_prod_type["puc_general_category"] = df_prod_type["puc_general_category"].apply(
    lambda x: PUC_GENERAL_CATEGORY_FINAL_MANUAL.get(x, x)
    if isinstance(x, str) else x
)

print("Done with final cleanup.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")
print("Missing 'puc_product_type' values replaced.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

## Step 9: Additional PUC columns cleanup

- Singularization and canonical cleanup of `puc_product_type`.
- Replacing `and` with `&` in `puc_product_family` and `puc_product_type`.
- Normalizing apostrophe, prime and mojibake characters.
- Final unconditional (phrase-level) replacements in `puc_product_type`.
- Converting `puc_kind`, `puc_product_family` and `puc_product_type` to Title Case.
- Final cleanup and conditional (context-dependent) replacements for the PUC columns.

In [ ]:
# Title-casing specific phrases
# (The context-dependent puc_product_type replacements already ran in the previous cell.)
print("Title-casing specific phrases.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

df_spec_edit = df_prod_type

# Title-casing specific phrases in specific cols
# Regex pattern -> replacement, loaded from rules/title_map.csv
TITLE_MAP = rules.load_map("title_map")

# Specify cols
cols = [
    'puc_product_type',
    'puc_product_family',
    'puc_general_category'
]

# Title-case: apply the regex replacements to the unique values only, then map back
for col in cols:
    orig_uniq = df_spec_edit[col].dropna().unique()
    new_uniq = pd.Series(orig_uniq)
    for pattern, replacement in TITLE_MAP.items():
        new_uniq = new_uniq.str.replace(
            pattern,
            replacement,
            case=False,
            regex=True
        )
    df_spec_edit[col] = df_spec_edit[col].map(dict(zip(orig_uniq, new_uniq)))

print("Done title-casing specific phrases.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


## Step 10: Fill missing PUC values and save

Fills empty `puc_product_type` cells with `Unknown` and replaces all other missing values in the four PUC columns with `NoID`. The table is then sorted by `puc_kind`, `puc_general_category`, `puc_product_family` and `puc_product_type` in a single stable multi-key sort (the previous version sorted four times in a row, so only the last key, `puc_product_type`, determined the order) and written to CSV (`PRODTYPE_CSV`).

In [ ]:
# Fill empty puc_product_type cells with 'Unknown'
print("Replacing missing 'puc_product_type' values with 'Unknown'.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

cpdat_fltrd = df_spec_edit

cpdat_fltrd['puc_product_type'] = (
    cpdat_fltrd['puc_product_type']
    .replace('', pd.NA)
    .fillna('Unknown')
)

print("Replacing all other missing values in all PUC cols with 'NoID'.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

cols_to_noid = [
    "puc_kind",
    "puc_general_category",
    "puc_product_family",
    "puc_product_type"
]

cpdat_fltrd[cols_to_noid] = (
    cpdat_fltrd[cols_to_noid]
        .astype("string")
        .replace(r"^\s*$", pd.NA, regex=True)   # empty or whitespace-only
        .fillna("NoID")
)

print("Missing values replaced with 'Unknown' or 'NoID'.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

In [ ]:
# Write CPDat with replaced missing puc_product_type values to CSV file
print("Sorting CPDat.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Sort
cpdat_fltrd = cpdat_fltrd.sort_values(
    ["puc_kind", "puc_general_category", "puc_product_family", "puc_product_type"],
    kind="stable",
)

print("Writing CPDat to CSV.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Save CPDat AFTER the InChIKey, weight_fraction & name/keyword filtration
cpdat_fltrd.to_csv(PRODTYPE_CSV, index=False)

print(f"CPDat written to {PRODTYPE_CSV}.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

## Step 11: Create `puc_harmonized`, sort and save

Creates and populates the `puc_harmonized` column in the format `puc_kind | puc_general_category | puc_product_family | puc_product_type`. The table is sorted by `puc_harmonized` (single stable sort; the previous version sorted five times in a row, so only the last key determined the order) and written to CSV (`HARMONIZED_CSV`), the final output.

In [ ]:
# Create & populate 'puc_harmonized' col
print("Creating & populating 'puc_harmonized' col.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

cpdat_harm = cpdat_fltrd

HARMONIZED_SEPARATOR = " | "

# Create column if not present
if "puc_harmonized" not in cpdat_harm.columns:
    cpdat_harm["puc_harmonized"] = None

# Move puc_harmonized after puc_product_type
cols = list(cpdat_harm.columns)
if "puc_product_type" in cols and "puc_harmonized" in cols:
    cols.insert(
        cols.index("puc_product_type") + 1,
        cols.pop(cols.index("puc_harmonized"))
    )
    cpdat_harm = cpdat_harm[cols]

print("Created 'puc_harmonized' col.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Helper: replace missing (non-string / blank) with literal "NaN"
def harmonized_series(s):
    return s.where(s.map(lambda x: isinstance(x, str) and bool(x.strip())), "NaN")

# Populate puc_harmonized (vectorized string concatenation)
cpdat_harm["puc_harmonized"] = harmonized_series(cpdat_harm["puc_kind"]).str.cat(
    [
        harmonized_series(cpdat_harm[c])
        for c in ["puc_general_category", "puc_product_family", "puc_product_type"]
    ],
    sep=HARMONIZED_SEPARATOR,
)

print("Populated 'puc_harmonized' col.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

In [ ]:
# Write CPDat with replaced missing puc_product_type values to CSV file
print("Sorting CPDat.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Sort
cpdat_harm = cpdat_harm.sort_values("puc_harmonized", kind="stable")

print("Writing CPDat to CSV.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Save CPDat AFTER the InChIKey, weight_fraction & name/keyword filtration
cpdat_harm.to_csv(HARMONIZED_CSV, index=False)

print(f"CPDat (filtered & PUC populated & PUC harmonized) written to {HARMONIZED_CSV}.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

## Step 12: Descriptive statistics and summary tables

The next cells compute summary statistics on the final table (`cpdat_harm`). The first cell shows per-column dtype, unique and missing counts (`df_stats` from `inquire_common`) and creates the empty accumulators (sets, dicts, counters) that the following cells fill. The last two cells write the unique values of the PUC columns (`PUC_SUMMARY_CSV`) and one row per `cas` with row counts, number of distinct products and central weight fraction min/mean/max (`STATS_CSV`).

In [ ]:
# Statistics accumulators
print("Creating statistics accumulators.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

df_stat = cpdat_harm

_ = df_stats(df_stat, "Filtered CPDat")

total_rows = len(df_stat)

unique_products = set()
unique_raw_chems = set()
unique_curated_chems = set()

column_null_counts = {}
overall_values_per_column = {}
unique_values_per_column = {}

value_counts_limited = {}  # optional sample column

print("Statistics accumulators created.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

In [ ]:
# Column-level statistics
print("Calculating column-level statistics.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

for col in df_stat.columns:
    non_null = df_stat[col].dropna()

    unique_values_per_column[col] = set(non_null.unique())
    overall_values_per_column[col] = non_null.shape[0]
    column_null_counts[col] = df_stat[col].isna().sum()

print("Column-level statistics calculated.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

In [ ]:
# Global unique entities
print("Calculating global unique entities.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

unique_products.update(df_stat["product_title"].dropna().unique())
unique_raw_chems.update(df_stat["orig_name"].dropna().unique())
unique_curated_chems.update(df_stat["name"].dropna().unique())

print("Global unique entities calculated.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

In [ ]:
# cas → product/weight fraction relationships
print("Establishing cas & product/weight fraction relationships.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# One row per cas, in order of first appearance (vectorized groupby)
casrn_stats = (
    df_stat.dropna(subset=["cas"])
    .groupby("cas", sort=False)
    .agg(
        name=("name", "first"),  # first non-null
        row_count=("cas", "size"),
        unique_products=("product_title", "nunique"),
        central_wf_mean=("central_weight_fraction", "mean"),
        central_wf_min=("central_weight_fraction", "min"),
        central_wf_max=("central_weight_fraction", "max"),
        lower_wf_min=("lower_weight_fraction", "min"),
        upper_wf_max=("upper_weight_fraction", "max"),
        lower_n=("lower_weight_fraction", "count"),
        upper_n=("upper_weight_fraction", "count"),
    )
)
casrn_stats["name"] = (
    casrn_stats["name"].fillna("Unknown").replace("", "Unknown")
)

print("Curated_casrn & product/weight fraction relationships established.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

In [ ]:
# Optional: value counts for a sample column
print("Counting values in product_type col.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

if "product_type" in df_stat.columns:
    vc = df_stat["product_type"].value_counts(dropna=False)
    value_counts_limited = vc.to_dict()

print("Done counting values in product_type col.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

In [ ]:
# Summary
print("===== OVERALL & UNIQUE VALUE COUNTS PER COLUMN =====\n")

# exclude non-important columns
exclude_cols = {"id_sort", "cpdat_data_document_id", "data_document_title", "data_document_subtitle", "data_source",
                "data_source_url", "data_group", "data_group_url", "data_group_download_date",
                "data_document_type", "data_document_url", "data_document_date", "organization",
                "puc_classification_method", "provisional_dtxsid", "weight_fraction_type", "component_name"}

rows = []

for col in unique_values_per_column:
    # if col in exclude_cols:
    #     continue

    rows.append({
        "Column": col,
        "Overall": overall_values_per_column.get(col, 0),
        "Unique": len(unique_values_per_column[col]),
        "Total rows": total_rows
    })

summary_df = pd.DataFrame(rows)

summary_df.style.format({
    "Overall": "{:,}",
    "Unique": "{:,}",
    "Total rows": "{:,}"
}).set_table_styles([
    {"selector": "th, td", "props": [("border", "1px solid black"), ("text-align", "left")]},
    {"selector": "table", "props": [("border-collapse", "collapse")]}
])

In [ ]:
# Show top N most frequent compounds
TOP_N = 30

top = casrn_stats.sort_values("row_count", ascending=False, kind="stable").head(TOP_N)

# WF bounds only reported when both lower and upper fractions exist
has_bounds = (top["lower_n"] > 0) & (top["upper_n"] > 0)

casrn_df = pd.DataFrame({
    "CASRN": top.index,
    "Chemical name": top["name"].to_numpy(),
    "Occurrences (rows)": top["row_count"].to_numpy(),
    "Unique products": top["unique_products"].to_numpy(),
    "Central WF min": top["central_wf_min"].to_numpy(),
    "Central WF max": top["central_wf_max"].to_numpy(),
    "Central WF mean": top["central_wf_mean"].to_numpy(),
    "WF bounds min": top["lower_wf_min"].where(has_bounds).to_numpy(),
    "WF bounds max": top["upper_wf_max"].where(has_bounds).to_numpy(),
})

# Render nicely with commas and borders
casrn_df.style.format({
    "Occurrences (rows)": "{:,}",
    "Unique products": "{:,}",
    "Central WF min": "{:.4g}",
    "Central WF max": "{:.4g}",
    "Central WF mean": "{:.4g}",
    "WF bounds min": "{:.4g}",
    "WF bounds max": "{:.4g}"
}).set_table_styles([
    {"selector": "th, td", "props": [("border", "1px solid black"), ("text-align", "left")]},
    {"selector": "table", "props": [("border-collapse", "collapse")]}
])

In [ ]:
# Write PUC stats to CSV file
print("Writing PUC stats to CSV.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Columns to summarize (the original notebook used `target_cols` without defining it; the PUC columns are assumed)
target_cols = ["puc_kind", "puc_general_category", "puc_product_family", "puc_product_type", "puc_harmonized"]

rows = []

for col in target_cols:
    if col not in unique_values_per_column:
        continue

    uniques = sorted(unique_values_per_column[col])
    unique_count = len(uniques)
    overall_count = overall_values_per_column.get(col, 0)

    for v in uniques:
        rows.append({
            "Column": col,
            "Overall": overall_count,
            "Unique": unique_count,
            "Total rows": total_rows,
            "Unique value": v
        })

# Build DataFrame
puc_df = pd.DataFrame(rows)

# Optional: sort for readability
puc_df = puc_df.sort_values(
    ["Column", "Unique value"],
    ignore_index=True
)

# Write to CSV
puc_df.to_csv(PUC_SUMMARY_CSV, index=False, encoding="utf-8")

print(f"PUC stats written to {PUC_SUMMARY_CSV}.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

In [ ]:
# Write stats DF to CSV file
print("Writing stats DF to CSV.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

stats_cols = [
    "cas", "name", "row_count", "unique_products",
    "central_wf_mean", "central_wf_min", "central_wf_max",
]
casrn_stats.reset_index()[stats_cols].to_csv(STATS_CSV, index=False, encoding="utf-8")

print(f"Stats DF written to {STATS_CSV}.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")